# SAADE–AUA v5 · 00 — Перевірка даних до навчання

**Призначення:** перевірити реальне походження 4 895 робочих аудіофайлів AUA, не включаючи архіви `00_sources_and_licenses`, дублікати `05_splits` та дані Мельника до навчальної вибірки. Це не запускає моделі й не відкриває hidden test.

**Два режими:** `pilot_existing` — інженерна перевірка методики на поточному AUA; `matched_tts` — основний експеримент після реально створених TTS-аудіо з тим самим текстом.

## 0. Імпорти та папки

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, shutil, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS = PROJECT / 'results'
OUT = RESULTS / 'SAADE_AUA_2026_v5_FINAL'
assert AUA.is_dir(), f'Не знайдено AUA: {AUA}'
assert shutil.which('ffmpeg'), 'Потрібно встановити ffmpeg, щоб відтворити MP3-сценарії'
sys.path.insert(0,str(Path.cwd().resolve()))
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A
for mod in (S,D,M,V,A):
    assert Path(mod.__file__).resolve().parent == Path.cwd().resolve(), f'Чужа версія модуля: {mod.__file__}'
cfg=S.Config(root=str(AUA),out=str(OUT),scenario_profile='v2_5level',
             seed=42,seconds=4.,min_seconds=2.,augmentation_probability=.50,
             batch_size=12,lr=.0005,epochs=15,use_cache=True)
print('AUA:',AUA,'\nRESULTS:',OUT)
print('Дані лише для читання. Melnyk_dus не використовується.')

## 1. Розпізнавання метаданих і контроль DRSSU

In [ ]:
catalog=D.build_verified_catalog(cfg)
display(pd.crosstab(catalog.source,[catalog.label,catalog.usable],dropna=False))
display(catalog.groupby(['source','identity_status','label_source'],dropna=False)
        .size().rename('files').reset_index())
print('Придатні:',int(catalog.usable.sum()),'/',len(catalog))
print('DRSSU verified via selection_rank:',int((catalog.source.eq('DRSSU') & catalog.label.eq(1)).sum()))
print('Проблемні записи:',(OUT/'01_catalog_issues.csv'))

## 2. Вихідний груповий поділ і джерело можливого змішування

In [ ]:
parents,split_report=D.prepare_parent_splits(catalog,cfg,
        max_parent_files=900,unknown_train_cap=150,policy='exploratory')
display(split_report)
display(D.data_readiness_report(catalog,parents,cfg,policy='exploratory'))
print('Незалежні групи за класами:')
display(parents.groupby(['split','label']).group_id.nunique().rename('groups').reset_index())
assert not parents.groupby('group_id').split.nunique().gt(1).any()

## 3. Черга для Text → TTS

Це **план**, а не згенерований звук. Зміст транскрипцій, ліцензії й дозвіл на генерацію потрібно перевірити; файли не генеруються самі собою.

In [ ]:
queue=A.generate_tts_job_queue(parents,cfg,max_real=400,seed=42)
display(queue.groupby('split').size().rename('texts').reset_index())
display(queue.head(10))
print('CSV для Text→TTS:',OUT/'V5_00_TTS_GENERATION_QUEUE.csv')
print('Наступний файл: 01_Text_to_TTS.ipynb')

## 4. Перелік очікуваних джерел обмежень

Якщо source повністю визначає real/fake або `voice_id` невідомий, це **методологічне обмеження**, а не доказ якості класифікації. Не можна штучно відновлювати голоси з технічних ID DRSSU/FLEURS.